# Build An Acceptance Gate That Survives Small Eval Sets

Companion notebook for the [Evaluations and Acceptance Tests](https://codemia.io/courses/forward_deployed_engineering/fde_evaluations_and_acceptance_tests) lesson on Codemia.

**What you will build**: the release gate an FDE agrees with a customer before an LLM system goes
live. You will simulate a labelled evaluation set split into customer segments, score a baseline
and a candidate version, put confidence intervals on every number with the bootstrap, compare
the two versions the paired way, work out how many cases you need to see a real difference,
correct for an automated judge that disagrees with human reviewers and finally write a gate
function that returns pass or fail with reasons a sponsor can read.

No model calls and no GPU: every system output is simulated, so the statistics are the point.
`numpy` and `matplotlib` only.

## Setup

In [ ]:
%pip install -q numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(11)

## Concept recap

An acceptance test turns "is it good enough" into numbers both sides signed in advance: a target
pass rate overall, a floor for each important segment and a rule that the new version must not
be worse than the old one. The trap is that eval sets in the field are small, often a few hundred
cases a customer expert labelled by hand, so every pass rate carries real uncertainty. A gate
that ignores that uncertainty either blocks good releases by chance or waves through regressions
that a larger set would have caught.

## A labelled eval set with segments

Four kinds of customer request, in the proportions the customer sees, with the probability that
each version answers a case correctly. These true rates are hidden in a real deployment; here we
know them so we can check whether the statistics recover them.

In [ ]:
SEGMENTS = {               # name: (cases in eval set, baseline true rate, candidate true rate)
    'billing questions': (180, 0.90, 0.93),
    'policy lookups':    (120, 0.86, 0.90),
    'claims status':     (70,  0.84, 0.86),
    'complaints':        (30,  0.80, 0.70),   # the candidate quietly got worse here
}

seg, base, cand = [], [], []
for name, (n, pb, pc) in SEGMENTS.items():
    # Paired outcomes: the same case is hard or easy for both versions. Draw a shared difficulty
    # and let each version pass when the difficulty is under its own rate.
    u = rng.random(n)
    seg += [name] * n
    base.append(u < pb)
    cand.append(u < pc)
seg = np.array(seg)
base = np.concatenate(base)
cand = np.concatenate(cand)
print(f'{len(seg)} cases   baseline pass {base.mean():.1%}   candidate pass {cand.mean():.1%}')
for name in SEGMENTS:
    m = seg == name
    print(f'  {name:<18} n={m.sum():>3}   baseline {base[m].mean():.1%}   candidate {cand[m].mean():.1%}')

Overall the candidate looks better, and it is. But one segment got worse, and that segment is the
one where a wrong answer reaches an angry customer. Whether the eval can show that is the
question for the rest of the notebook.

## Confidence intervals with the bootstrap

Resample the cases with replacement many times, recompute the pass rate each time and take the
middle 95 percent. It needs no formula and works for any statistic, including the segment rates
and the difference between versions.

In [ ]:
def boot_ci(x, stat=np.mean, n=4000, alpha=0.05):
    idx = rng.integers(0, len(x), size=(n, len(x)))
    vals = stat(x[idx], axis=1) if stat is np.mean else np.array([stat(x[i]) for i in idx])
    return np.quantile(vals, [alpha / 2, 1 - alpha / 2])

rows = [('overall', np.ones_like(seg, dtype=bool))] + [(n, seg == n) for n in SEGMENTS]
print(f'{"":<18} {"candidate":>9}   95% interval')
for name, m in rows:
    lo, hi = boot_ci(cand[m].astype(float))
    print(f'{name:<18} {cand[m].mean():>9.1%}   {lo:.1%} to {hi:.1%}   width {hi - lo:.1%}')

The overall interval is a few points wide. The complaints interval, from thirty cases, is wide
enough to hold almost any answer. A segment floor checked on thirty cases is mostly checking luck.

## Compare versions the paired way

Both versions ran on the same cases, so compare them case by case. Cases both pass or both fail
say nothing about the difference; only the discordant cases do. Treating the two runs as
independent samples throws that pairing away and makes the interval much wider.

In [ ]:
def diff_ci_paired(a, b, n=4000):
    idx = rng.integers(0, len(a), size=(n, len(a)))
    d = b[idx].mean(axis=1) - a[idx].mean(axis=1)
    return np.quantile(d, [0.025, 0.975])

def diff_ci_unpaired(a, b, n=4000):
    ia = rng.integers(0, len(a), size=(n, len(a)))
    ib = rng.integers(0, len(b), size=(n, len(b)))
    d = b[ib].mean(axis=1) - a[ia].mean(axis=1)
    return np.quantile(d, [0.025, 0.975])

b, c = base.astype(float), cand.astype(float)
print('discordant cases: candidate fixed', int((~base & cand).sum()), ' candidate broke', int((base & ~cand).sum()))
for label, fn in (('paired', diff_ci_paired), ('unpaired', diff_ci_unpaired)):
    lo, hi = fn(b, c)
    print(f'{label:<9} difference {c.mean() - b.mean():+.1%}   95% interval {lo:+.1%} to {hi:+.1%}')
m = seg == 'complaints'
lo, hi = diff_ci_paired(b[m], c[m])
print(f'complaints paired difference {c[m].mean() - b[m].mean():+.1%}   95% interval {lo:+.1%} to {hi:+.1%}')

The paired interval is narrower for the same data, which is why release comparisons should
always run both versions on the same cases. The complaints drop shows up here, but its interval
runs from a small slip to a collapse, and on thirty cases a real regression is often missed
entirely. The next experiment shows how often.

## How many cases do you need?

Simulate the eval many times at different sizes and count how often the paired interval for a
segment correctly lands entirely below zero when the candidate really is ten points worse. That
rate is the power of the test. Customers ask why the eval set needs more labelling; this chart is
the answer.

In [ ]:
def power(n, pb=0.80, pc=0.70, trials=300):
    hits = 0
    for _ in range(trials):
        u = rng.random(n)
        a, bb = (u < pb).astype(float), (u < pc).astype(float)
        lo, hi = diff_ci_paired(a, bb, n=600)
        hits += hi < 0
    return hits / trials

sizes = [20, 30, 50, 80, 120, 200]
pw = [power(n) for n in sizes]
for n, p in zip(sizes, pw):
    print(f'{n:>4} cases -> detects a ten point drop {p:.0%} of the time')

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(sizes, pw, marker='o')
ax.axhline(0.8, ls='--', color='gray', lw=0.8)
ax.set_xlabel('cases in the segment')
ax.set_ylabel('chance the gate catches it')
ax.set_ylim(0, 1.02)
ax.set_title('Thirty cases catch a ten point regression less than half the time')
plt.tight_layout()
plt.show()

With a shared difficulty between versions, a ten point drop shows up as ten percent of cases that
the old version passed and the new one fails, and nothing pulling the other way. Thirty cases
usually contain only a few of those. Ask the customer for enough labelled cases in each segment
they care about to reach the eighty percent line, and write the size into the acceptance plan.

## An automated judge that is not quite right

Grading every case by hand does not scale, so most teams use an automated judge, often another
model, and check it against humans on a sample. The judge has its own error rates: it passes
some wrong answers and fails some right ones. The raw judged pass rate is then biased, and the
bias can be corrected once you measure the judge.

In [ ]:
SENS, SPEC = 0.99, 0.55      # a lenient judge: passes 99% of correct answers, fails only 55% of wrong ones

def judge(truth):
    r = rng.random(len(truth))
    return np.where(truth, r < SENS, r > SPEC)

judged = judge(cand)
print(f'true candidate pass {cand.mean():.1%}   judged pass {judged.mean():.1%}')

# Humans grade a random sample of 150 cases. Estimate the judge error rates from it.
sample = rng.choice(len(cand), 150, replace=False)
t, j = cand[sample], judged[sample]
sens_hat = (j & t).sum() / t.sum()
spec_hat = (~j & ~t).sum() / max((~t).sum(), 1)
print(f'estimated sensitivity {sens_hat:.2f}   estimated specificity {spec_hat:.2f}   (from {(~t).sum()} wrong answers)')

# Correct the judged rate: observed = p*sens + (1-p)*(1-spec), solve for p.
p_obs = judged.mean()
p_corr = (p_obs + spec_hat - 1) / (sens_hat + spec_hat - 1)
print(f'corrected pass rate {p_corr:.1%}')

A lenient judge inflates the pass rate by several points, enough to clear a target the system
does not really meet. Measure the judge on a human labelled sample, report its agreement in the
acceptance results and correct the rate, or the gate is grading the judge rather than the system.
Notice also how few wrong answers the human sample contains: the specificity estimate rests on a
handful of cases, and it is the number that drives the bias. Sample more of the failures, or
label a larger sample, before trusting the correction.

## The gate

Put the agreed criteria in one function that returns a decision and the reasons. Keep the rules
in plain words so the sponsor can read the output without the notebook.

In [ ]:
CRITERIA = {
    'overall_target': 0.88,          # point estimate must reach this
    'overall_floor_ci': 0.85,        # and the lower bound must stay above this
    'max_segment_drop': 0.05,        # no segment may fall by more than this versus baseline
    'min_segment_cases': 80,         # below this a segment result is reported as not decidable
}

def gate(seg, base, cand, crit=CRITERIA):
    reasons, ok = [], True
    c = cand.astype(float)
    lo, _ = boot_ci(c)
    if c.mean() < crit['overall_target']:
        ok = False; reasons.append(f'FAIL overall {c.mean():.1%} below target {crit["overall_target"]:.0%}')
    elif lo < crit['overall_floor_ci']:
        ok = False; reasons.append(f'FAIL overall lower bound {lo:.1%} below {crit["overall_floor_ci"]:.0%}')
    else:
        reasons.append(f'pass overall {c.mean():.1%} (lower bound {lo:.1%})')
    for name in np.unique(seg):
        m = seg == name
        d = cand[m].mean() - base[m].mean()
        dlo, dhi = diff_ci_paired(base[m].astype(float), cand[m].astype(float))
        if m.sum() < crit['min_segment_cases']:
            reasons.append(f'UNDECIDED {name}: {m.sum()} cases, change {d:+.1%} ({dlo:+.1%} to {dhi:+.1%}); label more')
            if d < -crit['max_segment_drop']:
                ok = False; reasons[-1] = 'FAIL ' + reasons[-1][10:] + ', and the point estimate already exceeds the allowed drop'
        elif dhi < -crit['max_segment_drop'] or d < -crit['max_segment_drop']:
            ok = False; reasons.append(f'FAIL {name}: change {d:+.1%} ({dlo:+.1%} to {dhi:+.1%})')
        else:
            reasons.append(f'pass {name}: change {d:+.1%} ({dlo:+.1%} to {dhi:+.1%})')
    return ('ACCEPT' if ok else 'REJECT'), reasons

decision, reasons = gate(seg, base, cand)
print(decision)
for r in reasons:
    print('  ' + r)

The gate rejects the candidate even though it is better overall, because the agreed rules say a
segment may not regress, and it flags the small segments as needing more labelled cases rather
than pretending to know. That output is what goes into the acceptance report: the decision, each
rule and the evidence, in sentences the sponsor can forward.

## Exercises

1. Grow the complaints segment to 120 cases in `SEGMENTS` and rerun. Does the gate now reject it
   for a clear regression instead of an undecided one?
2. Replace the shared difficulty with independent draws for each version. What happens to the
   paired interval, and why is shared difficulty the realistic assumption?
3. Change the judge to a strict one, `SENS, SPEC = 0.90, 0.95`. Which way is the judged rate now
   biased, and how many human labels do you need before the correction is reliable?
4. Add a latency rule to the gate: p95 response time under four seconds, using simulated
   latencies. Which rules belong in the same gate and which belong in monitoring after go live?

## What to take away

- Every pass rate from a small eval set needs an interval. The bootstrap gives one for any number.
- Compare versions on the same cases, paired. It is far more sensitive than comparing two rates.
- Segment floors need enough cases per segment. Work out the size with a power check and put it
  in the acceptance plan.
- An automated judge has error rates. Measure them on a human sample and correct the result.
- Write the gate as code with readable reasons, so the decision is the same every time and the
  sponsor can see why.